In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, upper, current_timestamp, coalesce, lit, date_format
from pyspark.sql.types import IntegerType, DoubleType, StringType

StatementMeta(sparkpoolbank, 82, 2, Finished, Available, Finished, False)

In [6]:
bronze_path = "abfss://bronze@stbankingretail.dfs.core.windows.net/Customer_Master/*.csv"

df_bronze = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(bronze_path)

# df_bronze.show()

StatementMeta(sparkpoolbank, 82, 3, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import col, trim, current_timestamp, coalesce, lit, to_date, regexp_extract, when,date_format

from pyspark.sql.functions import col, trim, coalesce, lit, when, regexp_extract, to_date, current_timestamp, date_format

spark.conf.set("spark.sql.parquet.outputTimestampType", "INT96")
spark.conf.set("spark.sql.parquet.writelegacyformat", "true")

# 1. Transformations
df_silver = (
    df_bronze.withColumn("customer_id_raw", trim(col("Customer_Id")))
    .withColumn("CustomerID", trim(col("Customer_ID")))
    .withColumn("CustomerName", coalesce(trim(col("Name")), lit("Unknown")))
    .withColumn(
        "Gender",
        when(trim(col("Gender")).isin("M", "Male"), "Male")
        .when(trim(col("Gender")).isin("F", "Female"), "Female")
        .otherwise("Unknown"),
    )
    .withColumn("City", coalesce(trim(col("City")), lit("Unknown")))
    .withColumn(
        "CreditScore",
        when(
            (col("Credit_Score").cast("double") <= 0)
            | col("Credit_Score").isNull(),
            None,
        ).otherwise(col("Credit_Score").cast("double")),
    )
    .withColumn(
        "JoinDate",
        date_format(to_date(col("Join_Date"), "yyyy-MM-dd"), "yyyy-MM-dd").cast(
            StringType()
        ),
    )
    .withColumn(
        "IngestionDate",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss").cast(
            StringType()
        ),
    )
    .dropDuplicates(["customer_id_raw"])
)

# 2. Select columns with EXACT SQL Table Names
df_silver_final = df_silver.select(
    "CustomerID",
    "CustomerName",
    "Gender",
    "City",
    "CreditScore",
    "JoinDate",
    "IngestionDate",
)

# 3. Save as Parquet
silver_path = (
    "abfss://silver@stbankingretail.dfs.core.windows.net/Customer_Master"
)

df_silver_final.coalesce(1).write.format("parquet").option(
    "parquet.writer.version", "v1"
).mode("overwrite").option("overwriteSchema", "true").save(silver_path)

StatementMeta(sparkpoolbank, 82, 4, Finished, Available, Finished, False)

In [ ]:
# bronze_path = "abfss://silver@stbankingretail.dfs.core.windows.net/Customer_Master"

# df_silver = spark.read.format("parquet").load(silver_path)

# df_silver.printSchema()